# Assignment 11 — CNN Basics, Custom CNN & Transfer Learning

**Dataset:** Tiny ImageNet (64×64 RGB images)  
**Experiment subset:** 3 classes × 200 images = 600 images  
**Models:** Custom CNN, VGG16, MobileNet, ResNet50

> Run this notebook in Google Colab. The first setup cell downloads Tiny ImageNet (about 237 MB). Transfer-learning models use ImageNet weights and may download weights on first run. A GPU runtime is strongly recommended.

## 0. Environment setup

This notebook follows the assignment's requested 3-class, 600-image experiment. It chooses the first three class folders found in the Tiny ImageNet training directory. You can change `SELECTED_CLASSES` after inspecting the class names.

In [ ]:
import os, zipfile, shutil, random, urllib.request, time, warnings
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from tensorflow.keras import layers, models

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

## 1. Download and prepare Tiny ImageNet

In [ ]:
DATA_ROOT = Path("/content/tiny-imagenet-200")
ZIP_PATH = Path("/content/tiny-imagenet-200.zip")
URL = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"

if not DATA_ROOT.exists():
    if not ZIP_PATH.exists():
        print("Downloading Tiny ImageNet...")
        urllib.request.urlretrieve(URL, ZIP_PATH)
    print("Extracting dataset...")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall("/content")

TRAIN_DIR = DATA_ROOT / "train"
assert TRAIN_DIR.exists(), "Dataset extraction failed. Check the download URL and extracted folder."

all_classes = sorted([p.name for p in TRAIN_DIR.iterdir() if p.is_dir()])
print("Number of available classes:", len(all_classes))
print("First 20 class IDs:", all_classes[:20])

# Select any 3 classes. Replace these IDs if you prefer particular classes.
SELECTED_CLASSES = all_classes[:3]
print("Selected classes:", SELECTED_CLASSES)

### Create a balanced subset of exactly 600 images

Tiny ImageNet stores training images inside `train/<class-id>/images`. This cell collects up to 200 valid images per selected class and checks that the requested 600 images are available.

In [ ]:
SUBSET_DIR = Path("/content/tiny_imagenet_3class")
if SUBSET_DIR.exists():
    shutil.rmtree(SUBSET_DIR)
SUBSET_DIR.mkdir(parents=True)

valid_ext = {".jpg", ".jpeg", ".png"}
class_counts = {}

for class_id in SELECTED_CLASSES:
    src = TRAIN_DIR / class_id / "images"
    image_files = sorted([p for p in src.iterdir() if p.suffix.lower() in valid_ext])
    chosen = image_files[:200]
    dst = SUBSET_DIR / class_id
    dst.mkdir(parents=True)
    for i, path in enumerate(chosen):
        shutil.copy2(path, dst / f"{class_id}_{i:04d}{path.suffix.lower()}")
    class_counts[class_id] = len(chosen)

print("Images copied per class:", class_counts)
total_images = sum(class_counts.values())
print("Total images:", total_images)
assert total_images == 600, "Expected exactly 200 images in each of 3 classes."

## 2. Load images and create 70% / 15% / 15% split

In [ ]:
IMG_SIZE = (64, 64)
images, labels = [], []

for label, class_id in enumerate(SELECTED_CLASSES):
    folder = SUBSET_DIR / class_id
    for path in sorted(folder.iterdir()):
        try:
            img = tf.keras.utils.load_img(path, target_size=IMG_SIZE)
            arr = tf.keras.utils.img_to_array(img)
            images.append(arr)
            labels.append(label)
        except Exception as e:
            print("Skipped unreadable image:", path, e)

X = np.asarray(images, dtype=np.float32) / 255.0
y = np.asarray(labels, dtype=np.int64)
class_names = SELECTED_CLASSES

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Class distribution:", {class_names[i]: int((y == i).sum()) for i in range(3)})

# First split: 70% train, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
# Split temporary equally: 15% validation, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print("Train:", X_train.shape, "=", len(X_train)/len(X)*100, "%")
print("Validation:", X_val.shape, "=", len(X_val)/len(X)*100, "%")
print("Test:", X_test.shape, "=", len(X_test)/len(X)*100, "%")

In [ ]:
plt.figure(figsize=(12, 4))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(X_train[i])
    plt.title(class_names[y_train[i]])
    plt.axis("off")
plt.suptitle("Sample Tiny ImageNet images")
plt.tight_layout()
plt.show()

## Task 1 — CNN basics on a 3×3 matrix

The following cells demonstrate convolution, padding, ReLU, sigmoid, max pooling, min pooling, global average pooling, flattening, dense layers, batch normalization, dropout and early stopping.

In [ ]:
# Random 3x3 input matrix and a 2x2 convolution kernel
matrix = np.random.randn(3, 3)
kernel = np.array([[1, 0], [0, -1]], dtype=float)

print("Input matrix:\n", matrix)
print("\nKernel:\n", kernel)

def convolve2d_valid(image, kernel):
    h, w = image.shape
    kh, kw = kernel.shape
    output = np.zeros((h-kh+1, w-kw+1), dtype=float)
    for i in range(output.shape[0]):
        for j in range(output.shape[1]):
            output[i, j] = np.sum(image[i:i+kh, j:j+kw] * kernel)
    return output

conv = convolve2d_valid(matrix, kernel)
print("\nValid convolution output:\n", conv)

In [ ]:
# Padding P=1 adds one layer of zeros around the input
padded = np.pad(matrix, pad_width=1, mode="constant", constant_values=0)
conv_padded = convolve2d_valid(padded, kernel)

print("Padded matrix (P=1):\n", padded)
print("\nConvolution after padding:\n", conv_padded)

In [ ]:
def relu(x):
    return np.maximum(0, x)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

print("ReLU:\n", relu(conv_padded))
print("\nSigmoid:\n", sigmoid(conv_padded))

In [ ]:
# Pooling demonstration using 2x2 windows and stride 1
def pool2d(image, mode="max", pool_size=2, stride=1):
    h, w = image.shape
    out_h = (h - pool_size) // stride + 1
    out_w = (w - pool_size) // stride + 1
    output = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            window = image[i*stride:i*stride+pool_size, j*stride:j*stride+pool_size]
            if mode == "max":
                output[i, j] = np.max(window)
            elif mode == "min":
                output[i, j] = np.min(window)
            else:
                raise ValueError("mode must be 'max' or 'min'")
    return output

print("Max pooling:\n", pool2d(conv_padded, "max"))
print("\nMin pooling:\n", pool2d(conv_padded, "min"))
print("\nGlobal average pooling:", np.mean(conv_padded))

In [ ]:
# Flattening and fully connected (dense) layer
flattened = conv_padded.flatten()
print("Flattened vector:", flattened)
print("Flattened shape:", flattened.shape)

dense = layers.Dense(4, activation="relu")
dense_output = dense(tf.convert_to_tensor(flattened.reshape(1, -1), dtype=tf.float32))
print("Dense layer output:", dense_output.numpy())

In [ ]:
# Batch Normalization and Dropout examples
sample_batch = tf.random.normal((5, 4))
bn = layers.BatchNormalization()
drop = layers.Dropout(0.5)

bn_output = bn(sample_batch, training=True)
drop_output = drop(bn_output, training=True)

print("Input batch shape:", sample_batch.shape)
print("After Batch Normalization:", bn_output.shape)
print("After Dropout:", drop_output.shape)
print("Dropout randomly sets some activations to zero during training.")

In [ ]:
# Early stopping is a training callback. A small demonstration model is defined here.
demo_model = models.Sequential([
    layers.Input(shape=(10,)),
    layers.Dense(8, activation="relu"),
    layers.Dense(1)
])
demo_model.compile(optimizer="adam", loss="mse")
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
)
print("EarlyStopping callback:", early_stopping)
print("It stops training when validation loss does not improve for 3 epochs.")

## Task 2 — Custom CNN and overfitting analysis

A custom CNN is trained on the training set, monitored on the validation set, and evaluated once on the held-out test set. Early stopping restores the weights from the best validation-loss epoch.

In [ ]:
NUM_CLASSES = len(class_names)
BATCH_SIZE = 32
EPOCHS = 20

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(
    len(X_train), seed=SEED
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(
    BATCH_SIZE
).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(
    BATCH_SIZE
).prefetch(tf.data.AUTOTUNE)

def build_custom_cnn():
    model = models.Sequential([
        layers.Input(shape=(64, 64, 3)),
        layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(NUM_CLASSES, activation="softmax")
    ], name="Custom_CNN")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

custom_model = build_custom_cnn()
custom_model.summary()

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=4, restore_best_weights=True, verbose=1
)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1
)

custom_history = custom_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

In [ ]:
# Plot training and validation loss / accuracy
history = custom_history.history
epochs_range = range(1, len(history["loss"]) + 1)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, history["loss"], marker="o", label="Training loss")
plt.plot(epochs_range, history["val_loss"], marker="o", label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Custom CNN — Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history["accuracy"], marker="o", label="Training accuracy")
plt.plot(epochs_range, history["val_accuracy"], marker="o", label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Custom CNN — Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

### How to interpret the curves

- **Overfitting:** training loss continues to fall while validation loss starts rising; training accuracy is notably higher than validation accuracy.
- **Underfitting:** both training and validation performance remain poor.
- **Reasonable fit:** training and validation curves improve together and remain relatively close.

Use the actual curves produced by your run when writing your observation; do not assume a result before training.

## Task 3 — Transfer learning: VGG16, MobileNet and ResNet50

In [ ]:
# Transfer-learning inputs are resized to 160x160 for the ImageNet backbones.
# Original images remain 64x64 for the custom CNN.
TRANSFER_SIZE = (160, 160)

def make_transfer_arrays(X_data):
    resized = tf.image.resize(X_data, TRANSFER_SIZE).numpy()
    return resized.astype(np.float32)

X_train_tl = make_transfer_arrays(X_train)
X_val_tl = make_transfer_arrays(X_val)
X_test_tl = make_transfer_arrays(X_test)

print(X_train_tl.shape, X_val_tl.shape, X_test_tl.shape)

In [ ]:
def build_transfer_model(architecture):
    inputs = layers.Input(shape=(160, 160, 3))

    if architecture == "VGG16":
        preprocess = tf.keras.applications.vgg16.preprocess_input
        base = tf.keras.applications.VGG16(
            include_top=False, weights="imagenet", input_shape=(160, 160, 3)
        )
    elif architecture == "MobileNet":
        preprocess = tf.keras.applications.mobilenet_v2.preprocess_input
        base = tf.keras.applications.MobileNetV2(
            include_top=False, weights="imagenet", input_shape=(160, 160, 3)
        )
    elif architecture == "ResNet50":
        preprocess = tf.keras.applications.resnet50.preprocess_input
        base = tf.keras.applications.ResNet50(
            include_top=False, weights="imagenet", input_shape=(160, 160, 3)
        )
    else:
        raise ValueError("Unknown architecture")

    base.trainable = False
    x = layers.Lambda(preprocess, name="preprocessing")(inputs)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = models.Model(inputs, outputs, name=architecture)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

def dataset_for_transfer(X_data, y_data, training=False):
    ds = tf.data.Dataset.from_tensor_slices((X_data, y_data))
    if training:
        ds = ds.shuffle(len(X_data), seed=SEED)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_tl_ds = dataset_for_transfer(X_train_tl, y_train, training=True)
val_tl_ds = dataset_for_transfer(X_val_tl, y_val)
test_tl_ds = dataset_for_transfer(X_test_tl, y_test)

In [ ]:
def evaluate_model(model, dataset, y_true, model_name):
    probabilities = model.predict(dataset, verbose=0)
    y_pred = np.argmax(probabilities, axis=1)
    return {
        "Model": model_name,
        "Test Accuracy": accuracy_score(y_true, y_pred),
        "Precision (macro)": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall (macro)": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "F1 Score (macro)": f1_score(y_true, y_pred, average="macro", zero_division=0)
    }, y_pred

results = []
predictions = {}

custom_result, custom_pred = evaluate_model(custom_model, test_ds, y_test, "Custom CNN")
results.append(custom_result)
predictions["Custom CNN"] = custom_pred
print(custom_result)

In [ ]:
# Train each pretrained network. Frozen ImageNet feature extractor + new classifier.
TRANSFER_EPOCHS = 8
transfer_models = {}

for arch in ["VGG16", "MobileNet", "ResNet50"]:
    print("\n" + "="*60)
    print("Training", arch)
    model = build_transfer_model(arch)
    model.summary()

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=3, restore_best_weights=True, verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1
        )
    ]

    model.fit(
        train_tl_ds,
        validation_data=val_tl_ds,
        epochs=TRANSFER_EPOCHS,
        callbacks=callbacks,
        verbose=1
    )

    result, pred = evaluate_model(model, test_tl_ds, y_test, arch)
    results.append(result)
    predictions[arch] = pred
    transfer_models[arch] = model
    print("Test metrics:", result)

    tf.keras.backend.clear_session()

In [ ]:
# Compare all models in a table
import pandas as pd

results_df = pd.DataFrame(results)
display(results_df.sort_values("Test Accuracy", ascending=False).reset_index(drop=True))

In [ ]:
# Visual comparison of metrics
metric_cols = ["Test Accuracy", "Precision (macro)", "Recall (macro)", "F1 Score (macro)"]
plot_df = results_df.set_index("Model")[metric_cols]
plot_df.plot(kind="bar", figsize=(12, 5), ylim=(0, 1))
plt.ylabel("Score")
plt.title("Custom CNN vs Transfer Learning Models")
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
# Classification report and confusion matrix for each model
for model_name, y_pred in predictions.items():
    print("\n", "="*65)
    print("Classification report:", model_name)
    print(classification_report(
        y_test, y_pred, labels=list(range(NUM_CLASSES)),
        target_names=class_names, zero_division=0
    ))

    cm = confusion_matrix(y_test, y_pred, labels=list(range(NUM_CLASSES)))
    plt.figure(figsize=(4, 3))
    plt.imshow(cm)
    plt.title(f"Confusion Matrix — {model_name}")
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.xticks(range(NUM_CLASSES), class_names, rotation=45)
    plt.yticks(range(NUM_CLASSES), class_names)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            plt.text(j, i, cm[i, j], ha="center", va="center")
    plt.tight_layout()
    plt.show()

## 4. Results and observations

After all models finish, use the generated metrics table and plots to complete these observations:

1. Which model achieved the highest test accuracy?
2. Which model achieved the highest macro precision, recall and F1 score?
3. Does the custom CNN show signs of overfitting or underfitting based on its training/validation curves?
4. How does transfer learning compare with training a CNN from scratch on only 600 images?
5. Mention any practical differences in training time and computational requirements.

**Important:** Results depend on the selected classes, random split, runtime/GPU and training duration. Report the values from your actual notebook execution.

## 5. Save trained custom model and results

In [ ]:
custom_model.save("/content/custom_cnn_assignment11.keras")
results_df.to_csv("/content/assignment11_model_comparison.csv", index=False)
print("Saved:")
print("/content/custom_cnn_assignment11.keras")
print("/content/assignment11_model_comparison.csv")

## Viva preparation — key concepts

- **Convolution:** slides a filter/kernel over an image to extract features such as edges and textures.
- **Padding (P=1):** adds one pixel of border (here zeros) around the input; helps preserve spatial dimensions depending on kernel and stride.
- **ReLU:** `max(0, x)`; introduces non-linearity and keeps positive activations.
- **Sigmoid:** maps a value to the range 0–1.
- **Max pooling:** keeps the maximum value in each window; emphasizes strongest activation.
- **Min pooling:** keeps the minimum value in each window.
- **Global average pooling:** averages each feature map into one value, reducing parameters compared with flattening.
- **Flatten:** converts a multi-dimensional feature tensor into a one-dimensional vector.
- **Dense layer:** connects each input feature to each output neuron.
- **Batch normalization:** normalizes activations during training to help stabilize and speed up learning.
- **Dropout:** randomly disables a fraction of units during training to reduce overfitting.
- **Early stopping:** stops training when a monitored validation metric stops improving.
- **Transfer learning:** reuses features learned by a model pretrained on a large dataset, then trains a new classifier for the target classes.
- **Accuracy:** fraction of correct predictions.
- **Precision:** of the samples predicted as a class, fraction that truly belong to it.
- **Recall:** of the samples belonging to a class, fraction correctly found.
- **F1 score:** harmonic mean of precision and recall.